# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KijoSal-dev/flyrank-ml-internship-wk1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: define the baseline rule and reason codes

reason_codes = {
    "CTR_OPPORTUNITY": "Impressions are present but clicks are relatively low.",
    "RANKING_OPPORTUNITY": "Average search position suggests the content may have ranking opportunity.",
    "ENGAGEMENT_OPPORTUNITY": "Available engagement or organic-session signals suggest an opportunity.",
    "MONITOR": "No strong baseline action signal was identified."
}

print("Baseline rule:")
for code, description in reason_codes.items():
    print(f"- {code}: {description}")

print("\nReason codes:", list(reason_codes.keys()))


Baseline rule:
- CTR_OPPORTUNITY: Impressions are present but clicks are relatively low.
- RANKING_OPPORTUNITY: Average search position suggests the content may have ranking opportunity.
- ENGAGEMENT_OPPORTUNITY: Available engagement or organic-session signals suggest an opportunity.
- MONITOR: No strong baseline action signal was identified.

Reason codes: ['CTR_OPPORTUNITY', 'RANKING_OPPORTUNITY', 'ENGAGEMENT_OPPORTUNITY', 'MONITOR']


## 1. My rule and its reason codes

I will use a simple baseline action score to prioritize content items for review. The rule is designed to be explainable and decision-support rather than a prediction of guaranteed performance.

The rule will use observed March 2026 performance signals:

- **CTR opportunity:** content has impressions but relatively few clicks.
- **Ranking opportunity:** content has a relatively weak average search position.
- **Engagement opportunity:** GA4 engaged sessions or organic sessions are available and relatively low.
- **Monitor:** none of the stronger action conditions are met.

Each item will receive a reason code explaining why it was selected. The score is a prioritization signal, not proof that an item will improve after an action.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# ML-07 setup: reconnect to Hugging Face and recreate the March feature frame

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get the Hugging Face token from the environment.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

# Connect DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

print("DuckDB connected.")
print("Rebuilding March 2026 feature frame...")


DuckDB connected.
Rebuilding March 2026 feature frame...


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Recreate the March 2026 feature frame

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_engaged_sessions,
        sessions_organic,
        ga4_data_available
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print("Feature frame created.")
print(f"Rows: {len(feature_frame):,}")
print("Columns:")
print(feature_frame.columns.tolist())


Feature frame created.
Rows: 9,841,378
Columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions', 'sessions_organic', 'ga4_data_available']


In [13]:
# Section 2: build the baseline ranked action queue

import os
import numpy as np
import pandas as pd

score_data = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_engaged_sessions",
        "sessions_organic",
    ]
].copy()

# CTR: only calculate where impressions are available and greater than zero.
score_data["ctr"] = np.where(
    score_data["gsc_impressions"] > 0,
    score_data["gsc_clicks"] / score_data["gsc_impressions"],
    np.nan
)

# Lower CTR = higher CTR opportunity.
score_data["ctr_opportunity"] = np.nan

ctr_mask = score_data["ctr"].notna()

score_data.loc[ctr_mask, "ctr_opportunity"] = (
    1 - score_data.loc[ctr_mask, "ctr"].rank(pct=True)
)

# Higher average position number = weaker observed ranking.
score_data["ranking_opportunity"] = np.nan

position_mask = score_data["gsc_avg_position"].notna()

score_data.loc[position_mask, "ranking_opportunity"] = (
    score_data.loc[position_mask, "gsc_avg_position"].rank(pct=True)
)

# Only use engagement where GA4 data is actually available.
score_data["engagement_opportunity"] = np.nan

engagement_mask = score_data["ga4_engaged_sessions"].notna()

score_data.loc[engagement_mask, "engagement_opportunity"] = (
    1
    - score_data.loc[engagement_mask, "ga4_engaged_sessions"].rank(pct=True)
)

# Combine the available signals.
score_data["action_score"] = (
    0.40 * score_data["ctr_opportunity"].fillna(0)
    + 0.40 * score_data["ranking_opportunity"].fillna(0)
    + 0.20 * score_data["engagement_opportunity"].fillna(0)
)

# Default action.
score_data["reason_code"] = "MONITOR"

# Assign reason codes based on strong directional signals.
score_data.loc[
    score_data["ranking_opportunity"] >= 0.75,
    "reason_code"
] = "RANKING_OPPORTUNITY"

score_data.loc[
    score_data["ctr_opportunity"] >= 0.75,
    "reason_code"
] = "CTR_OPPORTUNITY"

score_data.loc[
    score_data["engagement_opportunity"] >= 0.75,
    "reason_code"
] = "ENGAGEMENT_OPPORTUNITY"

# Rank highest score first.
score_data = score_data.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

score_data["rank"] = np.arange(1, len(score_data) + 1)

# Select final output columns.
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "sessions_organic",
]

baseline_queue = score_data[output_cols].copy()

# Save the required CSV.
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print(f"Rows ranked: {len(baseline_queue):,}")
print(f"Output file: {output_path}")

print("\nReason-code counts:")
print(baseline_queue["reason_code"].value_counts())

print("\nTop 20:")
baseline_queue.head(20)


Rows ranked: 9,841,378
Output file: work/outputs/baseline_action_score.csv

Reason-code counts:
reason_code
MONITOR                8938642
RANKING_OPPORTUNITY     902736
Name: count, dtype: int64

Top 20:


,rank,client_hash_id,content_hash_id,action_score,reason_code,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic
0,1,client_23a62021009f63c4,content_13ef8874a9a1ef5e,0.723540,RANKING_OPPORTUNITY,1,0,0.0,497.0,0,0
1,2,client_23a62021009f63c4,content_aacb637aa920b8ea,0.723540,RANKING_OPPORTUNITY,1,0,0.0,495.0,0,0
2,3,client_23a62021009f63c4,content_c7ebdf81f488f0d8,0.723539,RANKING_OPPORTUNITY,1,0,0.0,480.0,0,0
3,4,client_20259bd6705d81d4,content_8c34799f566ce23c,0.723539,RANKING_OPPORTUNITY,1,0,0.0,469.0,0,0
4,5,client_23a62021009f63c4,content_6f6a949b6e7eb3ae,0.723539,RANKING_OPPORTUNITY,1,0,0.0,465.0,0,0
5,6,client_e547b89c05043229,content_aa376cef98a5fae8,0.723539,RANKING_OPPORTUNITY,1,0,0.0,447.0,0,0
6,7,client_23a62021009f63c4,content_070944208ef3a890,0.723539,RANKING_OPPORTUNITY,1,0,0.0,445.0,0,0
7,8,client_23a62021009f63c4,content_74e1dddeda79c23c,0.723539,RANKING_OPPORTUNITY,1,0,0.0,444.0,0,0
8,9,client_20259bd6705d81d4,content_9afdc38dbabc43c6,0.723539,RANKING_OPPORTUNITY,2,0,0.0,403.0,0,0
9,10,client_23a62021009f63c4,content_adb05a2cdd05709d,0.723539,RANKING_OPPORTUNITY,1,0,0.0,383.0,0,0


## 2. Build the ranked queue

I built a baseline action score using the March 2026 feature data and ranked all 9,841,378 rows from highest to lowest score.

The ranked queue was saved to `work/outputs/baseline_action_score.csv` as required. The scoring rule assigns each content item a baseline reason code based on observable performance signals.

The output contained 8,938,642 rows classified as `MONITOR` and 902,736 classified as `RANKING_OPPORTUNITY`. No rows were classified as `CTR_OPPORTUNITY` or `ENGAGEMENT_OPPORTUNITY` by the implemented scoring thresholds.

The Top-20 review shows that the highest-ranked items generally have very low impression counts (mostly 1–2 impressions) and very poor reported average positions. This is an important limitation of the baseline score: extreme ranking values can receive a high score even when the content has very little search volume.

Therefore, I treat this ranking as a **directional decision-support queue**, not as proof that these are the highest-value pages to change. The Top-20 results need further review for volume, data quality, and practical usefulness.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Top-20 review
# Reload the saved ranked queue produced in Section 2.

import pandas as pd

output_path = "work/outputs/baseline_action_score.csv"

ranked_data = pd.read_csv(output_path)

# Take the 20 highest-ranked rows
top20 = ranked_data.sort_values("rank").head(20).copy()


def confidence_note(row):
    if row["gsc_impressions"] < 10:
        return "Low confidence: very low impression volume"
    elif row["gsc_impressions"] < 100:
        return "Moderate confidence: limited impression volume"
    else:
        return "Higher confidence: sufficient impression volume"


def what_could_be_wrong(row):
    if row["gsc_impressions"] < 10:
        return "Low volume may make the ranking signal unstable"
    elif pd.isna(row["gsc_avg_position"]):
        return "Search position is missing"
    elif row["gsc_avg_position"] > 100:
        return "Extreme position value may reduce practical usefulness"
    else:
        return "Baseline signals may not capture business context"


top20["confidence_note"] = top20.apply(confidence_note, axis=1)

top20["what_could_be_wrong"] = top20.apply(
    what_could_be_wrong,
    axis=1
)

review_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "reason_code",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "sessions_organic",
    "confidence_note",
    "what_could_be_wrong"
]

# Keep only columns that actually exist in the saved CSV
review_cols = [col for col in review_cols if col in top20.columns]

top20_review = top20[review_cols]

print("Number of rows reviewed:", len(top20_review))
print("\nTop-20 review:")
top20_review


Number of rows reviewed: 20

Top-20 review:


,rank,client_hash_id,content_hash_id,action_score,reason_code,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_engaged_sessions,sessions_organic,confidence_note,what_could_be_wrong
0,1,client_23a62021009f63c4,content_13ef8874a9a1ef5e,0.723540,RANKING_OPPORTUNITY,1,0,0.0,497.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
1,2,client_23a62021009f63c4,content_aacb637aa920b8ea,0.723540,RANKING_OPPORTUNITY,1,0,0.0,495.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
2,3,client_23a62021009f63c4,content_c7ebdf81f488f0d8,0.723539,RANKING_OPPORTUNITY,1,0,0.0,480.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
3,4,client_20259bd6705d81d4,content_8c34799f566ce23c,0.723539,RANKING_OPPORTUNITY,1,0,0.0,469.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
4,5,client_23a62021009f63c4,content_6f6a949b6e7eb3ae,0.723539,RANKING_OPPORTUNITY,1,0,0.0,465.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
5,6,client_e547b89c05043229,content_aa376cef98a5fae8,0.723539,RANKING_OPPORTUNITY,1,0,0.0,447.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
6,7,client_23a62021009f63c4,content_070944208ef3a890,0.723539,RANKING_OPPORTUNITY,1,0,0.0,445.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
7,8,client_23a62021009f63c4,content_74e1dddeda79c23c,0.723539,RANKING_OPPORTUNITY,1,0,0.0,444.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
8,9,client_20259bd6705d81d4,content_9afdc38dbabc43c6,0.723539,RANKING_OPPORTUNITY,2,0,0.0,403.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable
9,10,client_23a62021009f63c4,content_adb05a2cdd05709d,0.723539,RANKING_OPPORTUNITY,1,0,0.0,383.0,0.0,0.0,Low confidence: very low impression volume,Low volume may make the ranking signal unstable


## 3. Top-20 review

I reviewed the 20 highest-ranked items from the baseline action queue.

All 20 items were classified as `RANKING_OPPORTUNITY`. However, the results show an important limitation of the baseline score: every item has either 1 or 2 impressions, and all have 0 clicks. Their reported average positions range from 339 to 497.

Because the impression volume is extremely low, I marked all 20 items as low confidence. A single observation can produce an extreme position value, so these rankings should not be interpreted as strong evidence that the content is a high-priority optimization target.

The main reason these items ranked highly appears to be the extreme average-position signal rather than meaningful search volume. The baseline queue is therefore useful as a directional screening tool, but the Top-20 results would need additional volume thresholds and stability checks before being used for an actual content decision.

What could make these recommendations wrong includes very low observation volume, unstable average-position measurements, or missing business/context information that is not represented in the baseline features.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Weak picks + leakage check

# Identify weak picks in the Top-20.
# These are items with extremely low impression volume.
weak_picks = top20_review[
    top20_review["gsc_impressions"] < 10
].copy()

print("Weak picks in Top-20 (impressions < 10):", len(weak_picks))

print("\nWeak-pick summary:")
print(
    weak_picks[
        [
            "rank",
            "reason_code",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "confidence_note"
        ]
    ]
)

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

selected_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "sessions_organic"
]

# Fields that should NOT be used as predictive features.
forbidden_fields = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "label",
    "target",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

# Product / future-window fields that should not enter the baseline.
product_or_future_terms = [
    "product",
    "future",
    "next_",
    "next",
    "forecast",
    "outcome",
    "label",
    "target",
    "trend"
]

accidental_forbidden = [
    col for col in selected_features
    if col in forbidden_fields
]

accidental_product_future = [
    col for col in selected_features
    if any(term in col.lower() for term in product_or_future_terms)
]

print("\nSelected features:")
print(selected_features)

print("\nForbidden fields accidentally used:")
print(accidental_forbidden)

print("\nProduct/future/label-related fields accidentally used:")
print(accidental_product_future)

# Check the actual columns in the saved ranked queue.
saved_columns = pd.read_csv(
    "work/outputs/baseline_action_score.csv",
    nrows=5
).columns.tolist()

print("\nColumns in saved baseline output:")
print(saved_columns)

print("\nLeakage check:")
if not accidental_forbidden and not accidental_product_future:
    print("PASS: no forbidden, product, future-window, or label-derived fields were selected.")
else:
    print("REVIEW REQUIRED: potentially problematic fields were selected.")

# Final weak-pick conclusion
print("\nWeak-pick conclusion:")
if len(weak_picks) == 20:
    print(
        "All Top-20 picks have fewer than 10 impressions, "
        "so the entire Top-20 should be treated as low-confidence."
    )
else:
    print(
        f"{len(weak_picks)} of the Top-20 picks have fewer than 10 impressions."
    )



Weak picks in Top-20 (impressions < 10): 20

Weak-pick summary:
    rank          reason_code  gsc_impressions  gsc_clicks  gsc_avg_position  \
0      1  RANKING_OPPORTUNITY                1           0             497.0   
1      2  RANKING_OPPORTUNITY                1           0             495.0   
2      3  RANKING_OPPORTUNITY                1           0             480.0   
3      4  RANKING_OPPORTUNITY                1           0             469.0   
4      5  RANKING_OPPORTUNITY                1           0             465.0   
5      6  RANKING_OPPORTUNITY                1           0             447.0   
6      7  RANKING_OPPORTUNITY                1           0             445.0   
7      8  RANKING_OPPORTUNITY                1           0             444.0   
8      9  RANKING_OPPORTUNITY                2           0             403.0   
9     10  RANKING_OPPORTUNITY                1           0             383.0   
10    11  RANKING_OPPORTUNITY                1          

## 4. Weak picks + leakage check

The weak-pick check found that all 20 of the Top-20 recommendations have fewer than 10 impressions. Nineteen items have 1 impression and one item has 2 impressions. All 20 have 0 clicks and are classified as `RANKING_OPPORTUNITY`.

Because the impression volume is extremely low, I consider the entire Top-20 to be low-confidence. The extreme average-position values may be driving the high scores without providing enough search volume to support a strong action recommendation.

I also checked the selected features for leakage. The selected features were `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_engaged_sessions`, and `sessions_organic`. No forbidden identifier, label-derived, product-related, or future-window fields were accidentally selected.

The leakage check therefore passed. However, the baseline still has a practical limitation: it should not be treated as a final prioritization system without minimum-volume or stability thresholds.

Overall, the baseline is best viewed as a directional decision-support tool. The Top-20 results demonstrate why human review and additional safeguards are needed before taking action.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.